# Collections avancées : `ChainMap`, `deque`, `defaultdict` et `Counter`, mesures avec `timeit`

## Fusion de configuration

Une application lit sa configuration à trois niveaux : ses valeurs par défaut (`defauts`), le fichier de l'utilisateur (`utilisateur`) et les options de la ligne de commande (`ligne_de_commande`). Chaque niveau l'emporte sur les précédents.

1. Avec `collections.ChainMap`, construisez une vue `configuration` des trois niveaux, puis affichez `dict(configuration)`. Attendu : `{'taille_police': 11, 'marge_gauche': 0, 'marge_droite': 2, 'marge_haute': 3}`.
2. Construisez aussi, avec l'opérateur `|`, un dictionnaire `fusion` aux mêmes valeurs. L'utilisateur passe ensuite sa marge droite à 4 : modifiez `utilisateur`, puis affichez `marge_droite` dans `configuration` et dans `fusion`. Attendu : `4 2`.
3. Avec `new_child()`, ajoutez à `configuration` une surcharge temporaire de la taille de police à 14, dans une nouvelle vue `temporaire`, puis affichez la taille de police dans `temporaire` et dans `configuration`. Attendu : `14 11`.

**Documentation utile**

- [`ChainMap.new_child`](https://docs.python.org/fr/3/library/collections.html#collections.ChainMap.new_child)

In [ ]:
defauts = dict(taille_police=10, marge_gauche=1, marge_droite=2)
utilisateur = dict(taille_police=11, marge_haute=3)
ligne_de_commande = dict(marge_gauche=0)


# Votre code ici

### Conseils supplémentaires

`ChainMap(premier, deuxieme, ...)` cherche une clef dans ses dictionnaires dans l'ordre des arguments et rend la première trouvée : le niveau qui l'emporte vient donc en premier, ici `ligne_de_commande`. Avec `a | b`, c'est l'inverse : la valeur de `b`, le dernier opérande, l'emporte, et le résultat est un nouveau dictionnaire. `vue.new_child(d)` renvoie un nouveau `ChainMap` où `d` passe devant tous les dictionnaires de `vue`, sans modifier `vue`.

```python
import collections


configuration = collections.ChainMap(...)
print(dict(configuration))

fusion = ...
utilisateur["marge_droite"] = 4
print(configuration["marge_droite"], fusion["marge_droite"])

temporaire = configuration.new_child(...)
print(temporaire["taille_police"], configuration["taille_police"])
```

### Solution

In [ ]:
import collections


configuration = collections.ChainMap(ligne_de_commande, utilisateur, defauts)
print(dict(configuration))

fusion = defauts | utilisateur | ligne_de_commande
utilisateur["marge_droite"] = 4
print(configuration["marge_droite"], fusion["marge_droite"])

temporaire = configuration.new_child(dict(taille_police=14))
print(temporaire["taille_police"], configuration["taille_police"])

`ChainMap` cherche chaque clef dans les dictionnaires dans l'ordre où on les lui a donnés, d'où la priorité de la ligne de commande sur l'utilisateur, et de l'utilisateur sur les défauts. C'est une vue, pas une copie : la modification de `utilisateur` s'y voit aussitôt, alors que `fusion`, construit par `|` (où le dernier opérande l'emporte), a gardé les valeurs du moment de la fusion. `new_child()` ajoute un niveau devant les autres sans toucher à `configuration`. L'itération (`items()`, `dict(configuration)`), elle, parcourt les dictionnaires du dernier au premier : les clefs des défauts sortent donc d'abord (`taille_police`, `marge_gauche`, `marge_droite`), puis `marge_haute`.

## Choisir la bonne structure

1. Avec `timeit.timeit`, mesurez la durée de 1 000 tests d'appartenance (`-1 in valeurs`) quand `valeurs` est une liste de 100 000 entiers, puis quand c'est un ensemble.
2. Mesurez de même la durée de 10 000 retraits en tête sur 100 000 éléments : `pop(0)` sur une liste, `popleft()` sur une `collections.deque`.

Expliquez les écarts observés à l'aide de la complexité de chaque opération.

**Documentation utile**

- [`timeit.timeit`](https://docs.python.org/fr/3/library/timeit.html#timeit.timeit)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`timeit.timeit(stmt, setup=..., number=...)` exécute `number` fois `stmt` et renvoie la durée totale en secondes, sans compter `setup`, exécuté une fois avant. `stmt` est une fonction sans argument (une `lambda`) ou une chaîne de code. Pour les retraits en tête, chaque exécution vide un peu la liste : construisez-la dans `setup`, pour que sa construction ne soit pas chronométrée et que la mesure parte d'une liste pleine, et faites les 10 000 retraits dans une seule exécution (`number=1`). `setup` s'exécute dans un espace de noms à part : il importe lui-même ce que `stmt` utilise (`"import collections; ..."`).

```python
import collections
import timeit

valeurs_liste = list(range(100_000))
valeurs_ensemble = ...
print("liste :", timeit.timeit(lambda: -1 in valeurs_liste, number=1_000))
print("ensemble :", ...)

print(
  "list.pop(0) :",
  timeit.timeit(
    "for _ in range(10_000): valeurs.pop(0)",
    setup="valeurs = list(range(100_000))",
    number=1,
  ),
)
print("deque.popleft() :", ...)
```

### Solution

In [ ]:
import collections
import timeit

valeurs_liste = list(range(100_000))
valeurs_ensemble = set(valeurs_liste)
print("liste :", timeit.timeit(lambda: -1 in valeurs_liste, number=1_000))
print("ensemble :", timeit.timeit(lambda: -1 in valeurs_ensemble, number=1_000))

print(
  "list.pop(0) :",
  timeit.timeit(
    "for _ in range(10_000): valeurs.pop(0)",
    setup="valeurs = list(range(100_000))",
    number=1,
  ),
)
print(
  "deque.popleft() :",
  timeit.timeit(
    "for _ in range(10_000): valeurs.popleft()",
    setup="import collections; valeurs = collections.deque(range(100_000))",
    number=1,
  ),
)

Un test d'appartenance parcourt toute la liste ($\mathcal{O}(n)$), alors qu'un ensemble trouve la valeur par son hash ($\mathcal{O}(1)$) : plusieurs ordres de grandeur d'écart. De même, `pop(0)` décale tous les éléments restants de la liste ($\mathcal{O}(n)$), alors que `popleft()` retire l'élément de tête d'une `deque` en $\mathcal{O}(1)$.

## Dernières lignes d'un fichier

Écrivez une fonction `dernieres_lignes(chemin, n)` qui renvoie les `n` dernières lignes d'un fichier, comme la commande Unix `tail`, sans jamais garder plus de `n` lignes en mémoire : le fichier peut être bien plus grand que la mémoire disponible. Utilisez une `collections.deque`.

Testez-la sur le fichier créé par la cellule suivante : `dernieres_lignes("nombres.txt", 3)` doit renvoyer `['99997\n', '99998\n', '99999\n']`.

In [ ]:
with open("nombres.txt", "w", encoding="utf8") as fichier:
  fichier.writelines(f"{i}\n" for i in range(100_000))

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un objet fichier ouvert en lecture s'itère ligne par ligne, chaque ligne gardant son `\n`. `collections.deque(iterable, maxlen=n)` consomme l'itérable en ne gardant que ses `n` derniers éléments : passez-lui directement le fichier, sans `readlines()`, qui chargerait tout le fichier en mémoire. `list(...)` en fait la liste à renvoyer.

```python
import collections


def dernieres_lignes(chemin: str, n: int) -> list[str]:
  with open(chemin, encoding="utf8") as fichier:
    return list(collections.deque(..., maxlen=...))


dernieres_lignes("nombres.txt", 3)
```

### Solution

In [ ]:
import collections


def dernieres_lignes(chemin: str, n: int) -> list[str]:
  with open(chemin, encoding="utf8") as fichier:
    return list(collections.deque(fichier, maxlen=n))


dernieres_lignes("nombres.txt", 3)

Une `deque` de taille maximale `n` retire automatiquement l'élément le plus ancien à chaque ajout : elle ne contient jamais plus de `n` lignes.

## Inventaire

On souhaite calculer le stock d'une boutique à partir d'une liste de transactions, des tuples nommés `Transaction(categorie, article, quantite)` où `quantite` est positive (achat) ou négative (vente).

1. En un seul parcours des transactions, calculez l'inventaire `inventaire`, le stock de chaque article de chaque catégorie, avec un `collections.defaultdict` de `collections.Counter`. Converti en dictionnaires ordinaires, il doit donner `{'nourriture': {'haricots': 4, 'maïs': 5}, 'vêtements': {'tshirt': 17}}`.
2. Avec une méthode de `Counter`, affichez le stock total de chaque catégorie. Attendu : `nourriture 9`, puis `vêtements 17`.

**Documentation utile**

- [`Counter.total`](https://docs.python.org/fr/3/library/collections.html#collections.Counter.total)

In [ ]:
from typing import NamedTuple


class Transaction(NamedTuple):
  categorie: str
  article: str
  quantite: int


transactions = [
  Transaction("nourriture", "haricots", 5),
  Transaction("nourriture", "maïs", 10),
  Transaction("vêtements", "tshirt", 20),
  Transaction("vêtements", "tshirt", -1),
  Transaction("nourriture", "maïs", -2),
  Transaction("nourriture", "haricots", -1),
  Transaction("vêtements", "tshirt", -1),
  Transaction("nourriture", "maïs", -3),
  Transaction("vêtements", "tshirt", -1),
]


# Votre code ici

### Conseils supplémentaires

`defaultdict(Counter)` crée un `Counter` vide la première fois qu'on lit une catégorie absente, et un `Counter` vaut `0` pour un article absent : `inventaire[categorie][article] += quantite` marche donc dès la première transaction, sans `if`. Un tuple nommé se dépaquette comme un tuple (`for categorie, article, quantite in transactions`). `{cle: dict(valeur) for cle, valeur in d.items()}` convertit l'inventaire en dictionnaires ordinaires, et `compteur.total()` renvoie la somme des comptes d'un `Counter`.

```python
from collections import Counter, defaultdict


inventaire = defaultdict(...)
for categorie, article, quantite in transactions:
  ...


print({categorie: dict(stock) for categorie, stock in inventaire.items()})
for categorie, stock in inventaire.items():
  print(categorie, ...)
```

### Solution

In [ ]:
from collections import Counter, defaultdict


inventaire = defaultdict(Counter)
for categorie, article, quantite in transactions:
  inventaire[categorie][article] += quantite


print({categorie: dict(stock) for categorie, stock in inventaire.items()})
for categorie, stock in inventaire.items():
  print(categorie, stock.total())

## Cache LRU (facultatif)

Un cache LRU (*least recently used*) garde au plus `taille` valeurs : plein, il fait de la place en retirant la valeur utilisée il y a le plus longtemps. Écrivez une classe `CacheLRU(taille)` qui range ses valeurs dans un `collections.OrderedDict`, `donnees`, dont les clefs vont de la moins récemment utilisée à la plus récente :

1. `lire(cle)` renvoie la valeur de `cle`, qui devient la plus récente, ou `None` si `cle` est absente ;
2. `ecrire(cle, valeur)` ajoute ou remplace la valeur de `cle`, qui devient la plus récente ; si le cache dépasse alors `taille` clefs, il retire la moins récente.

Testez-la sur un cache de taille 2 : écrivez `"a"` (1), puis `"b"` (2), lisez `"a"`, puis écrivez `"c"` (3). `"b"`, la clef la moins récemment utilisée, doit être sortie : `list(cache.donnees)` doit valoir `['a', 'c']`, et `cache.lire("b"), cache.lire("a"), cache.lire("c")` donner `None 1 3`.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`donnees.move_to_end(cle)` déplace une clef existante à la fin de l'`OrderedDict`, la place de la plus récente, et lève une `KeyError` si elle est absente : testez `cle in donnees` avant, dans `lire`. Une clef ajoutée va déjà à la fin, mais une clef remplacée garde sa place : dans `ecrire`, déplacez-la aussi. `donnees.popitem(last=False)` retire la première paire, la moins récente.

```python
import collections


class CacheLRU:
  def __init__(self, taille: int) -> None:
    self.taille = taille
    self.donnees = collections.OrderedDict()

  def lire(self, cle: str) -> int | None:
    if cle not in self.donnees:
      return None
    ...
    return self.donnees[cle]

  def ecrire(self, cle: str, valeur: int) -> None:
    self.donnees[cle] = valeur
    ...
    if len(self.donnees) > self.taille:
      ...


cache = CacheLRU(2)
cache.ecrire("a", 1)
cache.ecrire("b", 2)
cache.lire("a")
cache.ecrire("c", 3)
print(list(cache.donnees))
print(cache.lire("b"), cache.lire("a"), cache.lire("c"))
```

### Solution

In [ ]:
import collections


class CacheLRU:
  def __init__(self, taille: int) -> None:
    self.taille = taille
    self.donnees = collections.OrderedDict()

  def lire(self, cle: str) -> int | None:
    if cle not in self.donnees:
      return None
    self.donnees.move_to_end(cle)
    return self.donnees[cle]

  def ecrire(self, cle: str, valeur: int) -> None:
    self.donnees[cle] = valeur
    self.donnees.move_to_end(cle)
    if len(self.donnees) > self.taille:
      self.donnees.popitem(last=False)


cache = CacheLRU(2)
cache.ecrire("a", 1)
cache.ecrire("b", 2)
cache.lire("a")
cache.ecrire("c", 3)
print(list(cache.donnees))
print(cache.lire("b"), cache.lire("a"), cache.lire("c"))

L'ordre de l'`OrderedDict` est celui des usages : lire `"a"` l'a placée à la fin, si bien qu'à l'écriture de `"c"`, la première clef, celle que `popitem(last=False)` retire, est `"b"`. Chaque opération est en $\mathcal{O}(1)$, sans parcours des clefs. Pour garder en cache les résultats d'une fonction, `functools.lru_cache` fait la même chose en un décorateur.